# Lab Assignment 01: Predicting Diabetes with Deep Residual Multilayer Perceptron (Res-MLP)
**Clinical Binary Classification System with MICE Imputation, Cost-Sensitive Learning & Explainable AI**

### Clinical Project Overview
This notebook implements an elite-grade clinical machine learning and deep learning workflow for predicting whether a patient has diabetes mellitus using the Pima Indians Diabetes dataset.

**Key Clinical & Engineering Pillars:**
1. **Physiological Data Cleansing**: Identifies hidden biological zeroes and applies **Multivariate Imputation by Chained Equations (MICE)**.
2. **Evidence-Based Biomarker Engineering**: Calculates HOMA-IR Proxy, TyG Index, Insulin/Glucose Ratio, Age-BMI Interaction, and Metabolic Syndrome Severity.
3. **Multi-Model Benchmarking**: Benchmarks 5 conventional classifiers (Logistic Regression, KNN, SVM, Random Forest, Gradient Boosting) alongside the MLP.
4. **Deep Residual MLP (Res-MLP)**: Employs Swish activations, batch normalization, and skip connections with cost-sensitive class weighting.
5. **Youden's J Threshold Optimization**: Calibrates optimal screening cutoffs to maximize diagnostic sensitivity and slash False Negatives.
6. **Clinical Explainability**: Generates patient-level waterfall biomarker risk attributions.

In [ ]:
# 1. Setup and Library Imports
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, brier_score_loss
)

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
print('Environment and libraries initialized successfully!')

## 2. Dataset Understanding & Inspection (PDF Section 5)
We load the Pima Indians Diabetes dataset and inspect its structure, dimensions, data types, and target distribution.

In [ ]:
data_path = '../data/diabetes.csv' if os.path.exists('../data/diabetes.csv') else 'data/diabetes.csv'
df = pd.read_csv(data_path)
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
display(df.head())
display(df.describe().T)

## 3. Physiological Missing Values & Exploratory Data Analysis (PDF Section 6)
In clinical medicine, variables such as `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` cannot physiologically be zero in a living patient. These represent omitted laboratory tests.

In [ ]:
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
zero_summary = []
for col in zero_cols:
    count = (df[col] == 0).sum()
    pct = (count / len(df)) * 100
    zero_summary.append({'Feature': col, 'Zero Count': count, 'Percentage (%)': round(pct, 2)})

zero_df = pd.DataFrame(zero_summary)
print('Physiological Zero Anomalies (Hidden Missing Data):')
display(zero_df)

# Target Class Distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Outcome', palette=['#38bdf8', '#f43f5e'])
plt.title('Outcome Distribution (0: Non-Diabetic [65.1%], 1: Diabetic [34.9%])')
plt.show()
print('Class proportions:\n', df['Outcome'].value_counts(normalize=True))

## 4. Advanced Clinical Preprocessing: MICE & Biomarker Engineering (PDF Sections 7 & 8)
- Replace zeroes with `NaN` for physiological variables
- Stratified 80/20 train-test split executed **before** fitting imputer/scaler to prevent data leakage
- MICE (`IterativeImputer`) conditions missing variables on patient covariates
- Evidence-based biomarker engineering expands feature space from 8 to 17 variables
- Outlier-resilient scaling (`RobustScaler` + `StandardScaler`)

In [ ]:
# 1. Replace 0 with NaN
df_clean = df.copy()
for col in zero_cols:
    df_clean[col] = df_clean[col].replace(0, np.nan)

# 2. Stratified Split strictly before transformer fitting
X = df_clean.drop(columns=['Outcome'])
y = df_clean['Outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# 3. MICE Multivariate Imputer
mice_imputer = IterativeImputer(max_iter=20, random_state=42, initial_strategy='median', min_value=0.0)
X_train_imp = pd.DataFrame(mice_imputer.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_imp = pd.DataFrame(mice_imputer.transform(X_test), columns=X.columns, index=X_test.index)

# 4. Clinical Biomarker Engineering
def engineer_biomarkers(data):
    d = data.copy()
    d['HOMA_IR_Proxy'] = (d['Glucose'] * d['Insulin']) / 405.0
    d['TyG_Index'] = np.log(np.maximum((d['Glucose'] * d['BMI']) / 2.0, 1.0))
    d['Insulin_Glucose_Ratio'] = d['Insulin'] / (d['Glucose'] + 1.0)
    d['Age_BMI_Risk_Interaction'] = (d['Age'] * d['BMI']) / 100.0
    d['Glucose_BMI_Interaction'] = (d['Glucose'] * d['BMI']) / 100.0
    d['Metabolic_Syndrome_Score'] = (d['Glucose']>=100).astype(float) + (d['BMI']>=30).astype(float) + (d['BloodPressure']>=80).astype(float) + (d['Age']>=45).astype(float)
    d['BMI_Category'] = pd.cut(d['BMI'], bins=[-np.inf, 18.5, 24.9, 29.9, np.inf], labels=[0, 1, 2, 3]).astype(float)
    d['Glucose_Category'] = pd.cut(d['Glucose'], bins=[-np.inf, 99.9, 125.9, np.inf], labels=[0, 1, 2]).astype(float)
    d['Log_Insulin'] = np.log1p(d['Insulin'])
    return d

X_train_eng = engineer_biomarkers(X_train_imp)
X_test_eng = engineer_biomarkers(X_test_imp)

# 5. Robust + Standard Scaling
robust_scaler = RobustScaler()
standard_scaler = StandardScaler()

X_train_scaled = standard_scaler.fit_transform(robust_scaler.fit_transform(X_train_eng))
X_test_scaled = standard_scaler.transform(robust_scaler.transform(X_test_eng))
print(f'Feature dimension successfully expanded from 8 to {X_train_scaled.shape[1]} clinical biomarkers.')

## 5. Baseline Machine Learning Benchmarks (PDF Section 11)
Evaluating Logistic Regression, K-Nearest Neighbors, SVM, Random Forest, and Gradient Boosting.

In [ ]:
baselines = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7, weights='distance'),
    'Support Vector Machine': SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, max_depth=4, random_state=42)
}

results = []
for name, model in baselines.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall (Sensitivity)': round(recall_score(y_test, y_pred), 4),
        'F1-Score': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_prob), 4),
        'False Negatives (FN)': fn,
        'False Positives (FP)': fp
    })

display(pd.DataFrame(results))

## 6. Deep Residual Multilayer Perceptron (Res-MLP) Architecture & Cost-Sensitive Training
Building a Res-MLP featuring Swish activation, Batch Normalization, and a residual skip connection. Class weighting ($1.65\times$ positive penalty) penalizes missed diagnoses.

In [ ]:
# Validation split for MLP early stopping
X_tr_sub, X_val_sub, y_tr_sub, y_val_sub = train_test_split(
    X_train_scaled, y_train.values, test_size=0.15, stratify=y_train.values, random_state=42
)

# Functional Res-MLP Architecture
inputs = layers.Input(shape=(X_train_scaled.shape[1],))
x0 = layers.Dense(64, activation='swish', kernel_regularizer=regularizers.l2(0.0002))(inputs)
x0 = layers.BatchNormalization()(x0)
x0 = layers.Dropout(0.20)(x0)

res = layers.Dense(64, activation='swish')(x0)
res = layers.BatchNormalization()(res)
res = layers.Dropout(0.15)(res)
res = layers.Dense(64, activation='swish')(res)
res = layers.BatchNormalization()(res)

res_out = layers.Add()([x0, res])
x = layers.Dense(32, activation='swish')(res_out)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.10)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

res_mlp = keras.Model(inputs=inputs, outputs=outputs, name='Clinical_Res_MLP')
res_mlp.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), keras.metrics.Recall(name='recall')]
)

# Cost-Sensitive Class Weights
class_weights = {0: 0.77, 1: 1.65}

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=18, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-5)
]

history = res_mlp.fit(
    X_tr_sub, y_tr_sub,
    validation_data=(X_val_sub, y_val_sub),
    epochs=140,
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

## 7. Youden's J Optimization, Evaluation & Learning Curves
Evaluating the model at the standard 0.50 threshold versus Youden's optimal screening threshold ($J = \text{Sensitivity} + \text{Specificity} - 1$).

In [ ]:
# Predict probabilities on unseen test set
mlp_prob = res_mlp.predict(X_test_scaled).ravel()

# Compute Youden J optimal cutoff
fpr, tpr, thresholds = roc_curve(y_test, mlp_prob)
j_scores = tpr - fpr
opt_idx = np.argmax(j_scores)
opt_thresh = thresholds[opt_idx]
print(f'Youden Optimal Diagnostic Cutoff: {opt_thresh:.4f} (Default = 0.50)')

# Evaluate at 0.50 and Youden threshold
pred_std = (mlp_prob >= 0.50).astype(int)
pred_opt = (mlp_prob >= opt_thresh).astype(int)

cm_std = confusion_matrix(y_test, pred_std)
cm_opt = confusion_matrix(y_test, pred_opt)

results.append({
    'Model': 'Clinical Res-MLP (Standard 0.50)',
    'Accuracy': round(accuracy_score(y_test, pred_std), 4),
    'Precision': round(precision_score(y_test, pred_std), 4),
    'Recall (Sensitivity)': round(recall_score(y_test, pred_std), 4),
    'F1-Score': round(f1_score(y_test, pred_std), 4),
    'ROC-AUC': round(roc_auc_score(y_test, mlp_prob), 4),
    'False Negatives (FN)': cm_std.ravel()[2],
    'False Positives (FP)': cm_std.ravel()[1]
})
results.append({
    'Model': f'Clinical Res-MLP (Youden {opt_thresh:.2f})',
    'Accuracy': round(accuracy_score(y_test, pred_opt), 4),
    'Precision': round(precision_score(y_test, pred_opt), 4),
    'Recall (Sensitivity)': round(recall_score(y_test, pred_opt), 4),
    'F1-Score': round(f1_score(y_test, pred_opt), 4),
    'ROC-AUC': round(roc_auc_score(y_test, mlp_prob), 4),
    'False Negatives (FN)': cm_opt.ravel()[2],
    'False Positives (FP)': cm_opt.ravel()[1]
})

display(pd.DataFrame(results))

# Plot Learning Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['loss'], label='Train Loss', color='#0284c7')
ax1.plot(history.history['val_loss'], label='Val Loss', color='#f43f5e', linestyle='--')
ax1.set_title('Res-MLP Loss vs Epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Binary Crossentropy Loss')
ax1.legend()

ax2.plot(history.history['accuracy'], label='Train Accuracy', color='#10b981')
ax2.plot(history.history['val_accuracy'], label='Val Accuracy', color='#f59e0b', linestyle='--')
ax2.set_title('Res-MLP Accuracy vs Epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
plt.tight_layout()
plt.show()

## 8. Deployment Testing Across Clinical Scenarios (PDF Section 20)
Evaluating the model across four mandated patient cases: Healthy Adult, Diabetic, Borderline, and Extreme Boundary Case.

In [ ]:
def test_patient_profile(patient_dict, name):
    p_df = pd.DataFrame([patient_dict])
    p_imp = pd.DataFrame(mice_imputer.transform(p_df), columns=X.columns)
    p_eng = engineer_biomarkers(p_imp)
    p_scaled = standard_scaler.transform(robust_scaler.transform(p_eng))
    prob = float(res_mlp.predict(p_scaled, verbose=0)[0][0])
    label = 'Diabetic (1)' if prob >= opt_thresh else 'Non-Diabetic (0)'
    print(f'[{name}] Probability: {prob*100:.1f}% -> Decision: {label}')

test_patient_profile({'Pregnancies': 1, 'Glucose': 85.0, 'BloodPressure': 66.0, 'SkinThickness': 20.0, 'Insulin': 55.0, 'BMI': 21.4, 'DiabetesPedigreeFunction': 0.22, 'Age': 23}, 'Likely Non-Diabetic')
test_patient_profile({'Pregnancies': 7, 'Glucose': 185.0, 'BloodPressure': 90.0, 'SkinThickness': 40.0, 'Insulin': 280.0, 'BMI': 38.2, 'DiabetesPedigreeFunction': 0.95, 'Age': 55}, 'Likely Diabetic')
test_patient_profile({'Pregnancies': 3, 'Glucose': 128.0, 'BloodPressure': 78.0, 'SkinThickness': 28.0, 'Insulin': 110.0, 'BMI': 28.6, 'DiabetesPedigreeFunction': 0.45, 'Age': 40}, 'Borderline / Screen Positive')
test_patient_profile({'Pregnancies': 15, 'Glucose': 250.0, 'BloodPressure': 130.0, 'SkinThickness': 60.0, 'Insulin': 700.0, 'BMI': 55.0, 'DiabetesPedigreeFunction': 2.2, 'Age': 78}, 'Extreme Boundary Case')